Refer to 42.0 -> 42.2 (and monocyte 43.1)

https://nbviewer.org/github/yakirr/cna/blob/master/demo/demo.ipynb

# Header

In [1]:
from utils import *

In [2]:
import sys
sys.path.insert(0, '/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src')
import cna

In [3]:
os.chdir('../')

In [5]:
sc.settings.set_figure_params(dpi=600, dpi_save=600)  # 600 dpi for publication
sc.settings.figdir = "plots/Figure1"

rcParams.update({
    'font.family': 'Arial',
    'svg.fonttype': 'none',    # Keep SVG text as font
    'font.size': 6,            # Main font size (5–7pt recommended)
    'axes.titlesize': 7,
    'axes.labelsize': 6,
    'xtick.labelsize': 5,
    'ytick.labelsize': 5,
    'legend.fontsize': 6,
    'legend.title_fontsize': 6,
    'figure.dpi': 600,

    # --- Nature Immunology-style axes/ticks ---
    'axes.linewidth': 0.5,          # Axis lines: 0.5 pt
    'xtick.major.width': 0.5,        # Major tick lines
    'ytick.major.width': 0.5,
    'xtick.minor.width': 0.5,
    'ytick.minor.width': 0.5,
    'lines.linewidth': 0.7,          # Plot/data lines
    'axes.grid': False,              # No grid lines

    # --- Ticks on all sides ---
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.top': False,
    'xtick.bottom': False,
    'ytick.left': True,
    'ytick.right': False,


    # --- Tick label padding (minimal) ---
    'xtick.major.pad': 0,
    'ytick.major.pad': 0,

    # ---- Thin lines for Nature Immunology style ----
    'lines.linewidth': 0.5,     # General line width (default: 1.5)
    'patch.linewidth': 0.5,     # Outline for violin/bars (default: 1.0)
    
})

# plt.rcParams['axes.grid'] = False      # Extra guarantee, if using seaborn
# sns.set_style("white")                 # Removes upper/right axes and grid

# -- Scanpy vector-friendly mode --
sc.settings._vector_friendly = True

In [6]:
with open('data/Legends/celltypelevel3_palette.pkl','rb') as f:
    palette_all = pkl.load(f)

palette_all

{'Naive B': (0.843137, 0.0, 0.0),
 'Mature B': (0.54902, 0.235294, 1.0),
 'Memory B': (0.007843, 0.533333, 0.0),
 'Plasmablast': (0.0, 0.67451, 0.780392),
 'Naive CD4T': (0.596078, 1.0, 0.0),
 'Central Memory CD4T': (1.0, 0.498039, 0.819608),
 'Effector Memory CD4T': (0.423529, 0.0, 0.309804),
 'Cytotoxic CD4T': (1.0, 0.647059, 0.188235),
 'Treg': (0.345098, 0.231373, 0.0),
 'Naive CD8T': (0.0, 0.341176, 0.34902),
 'Central Memory CD8T': (0.0, 0.0, 0.866667),
 'Effector Memory CD8T': (0.0, 0.992157, 0.811765),
 'Double Negative T': (0.631373, 0.458824, 0.415686),
 'γδ T': (0.737255, 0.717647, 1.0),
 'MAIT': (0.584314, 0.709804, 0.470588),
 'cDC1': (0.752941, 0.015686, 0.72549),
 'cDC2': (0.392157, 0.329412, 0.454902),
 'pDC': (0.47451, 0.0, 0.0),
 'Classical Monocyte': (0.027451, 0.454902, 0.847059),
 'Intermediate Monocyte': (0.996078, 0.960784, 0.564706),
 'Nonclassical Monocyte': (0.0, 0.294118, 0.0),
 'Cytotoxic Monocyte': (0.560784, 0.478431, 0.0),
 'CD56-dim NK': (1.0, 0.447059, 

In [7]:
# major_celltype_palette = {
#     'B':    '#3072C4',   # strong blue
#     'T':    '#EF763F',   # bold orange
#     'DC':   '#35A293',   # teal
#     'Mono': '#B3813E',   # rich brown/gold
#     'NK':   '#7D4C99',   # deep purple
# }

cells_group = {
    'B':['Naive B',
 'Mature B',
 'Memory B',
 'Plasmablast'],
    'T':['Naive CD4T',
 'Central Memory CD4T',
 'Effector Memory CD4T',
 'Cytotoxic CD4T',
 'Treg',
 'Naive CD8T',
 'Central Memory CD8T',
 'Effector Memory CD8T',
 'Double Negative T',
 'γδ T',
 'MAIT'],
    'DC':[
   'cDC1',
 'cDC2',
 'pDC'     
    ],
    'Mono':[
        'Classical Monocyte',
 'Intermediate Monocyte',
 'Nonclassical Monocyte',
 'Cytotoxic Monocyte'
        ],
    'NK':[
        'CD56-dim NK',
 'CD56-bright NK',
 'Proliferating NK'
    ]
}

# Load data

In [13]:
meta = pd.read_csv('data/01_meta/meta_pt.csv')#, index_col=0)
#meta.index = [x.replace('_','-') for x in meta.index]

meta['race-stage'] = meta['race'].astype(str) + '-' + meta['group'].astype(str)

meta['race-stage'] = pd.Categorical(
    meta['race-stage'],
    categories=['NHW-HRL', 'NHW-DCIS', 'NHW-BC','NHB-HRL', 'NHB-DCIS', 'NHB-BC'],
    ordered=True
    )

meta

,sampleid,dataset_name,batch,viability_pct,group,race,age,diabetes,obesity,race-stage
0,9-D,Pool137_7,1,95,DCIS,NHW,37.0,N,N,NHW-DCIS
1,30-D,Pool137_12,1,92,DCIS,NHW,78.0,N,N,NHW-DCIS
2,36-D,Pool137_18,1,93,DCIS,NHB,39.0,N,N,NHB-DCIS
3,119-D,Pool137_20,1,95,DCIS,NHB,80.0,N,N,NHB-DCIS
4,28-D,Pool137_24,1,94,DCIS,NHW,53.0,N,N,NHW-DCIS
5,53-D,Pool137_27,1,96,DCIS,NHB,45.0,N,N,NHB-DCIS
6,77-D,Pool137_30,1,90,DCIS,NHW,45.0,N,N,NHW-DCIS
7,31-D,Pool138_6,2,93,DCIS,NHW,44.0,N,N,NHW-DCIS
8,6-H,Pool137_6,1,94,HRL,NHW,71.1,N,Y,NHW-HRL
9,1-H,Pool137_10,1,88,HRL,NHW,62.3,N,N,NHW-HRL


In [9]:
dat = sc.read_h5ad(
    'data/03_cleandata/ft_all.h5ad'
)

dat

AnnData object with n_obs × n_vars = 434565 × 27879
    obs: 'nFeaturess_RNA', 'nFeaturess_RNA_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublets_score', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'predicted.celltype.l1.score', 'predicted.celltype.l1', 'predicted.celltype.l2.score', 'predicted.celltype.l2', 'predicted.celltype.l3.score', 'predicted.celltype.l3', 'mapping.score', 'RNA_snn_res.0.8', 'seurat_clusters', 'dataset_name', 'RNA_snn_res.0.1', 'S.Score', 'G2M.Score', 'Phase', 'celltype_level3'
    var: 'vf_vst_counts.Pool137_10_mean', 'vf_vst_counts.Pool137_10_variance', 'vf_vst_counts.Pool137_10_variance.expected', 'vf_vst_counts.Pool137_10_variance.standardized', 'vf_vst_counts.Pool137_10_variable', 'vf_vst_counts.Pool137_10_rank', 'vf_vst_counts.Pool137_11_mean', 'vf_vst_counts.Pool137_11_variance', 'vf_vst_counts.Pool137_11_variance.expected', 'vf_vst_counts.Pool137_11_variance.standardized', 'vf_vst_counts.Pool137_11_variable', 'vf_vst_counts.Pool137

In [10]:
print(
    len(dat.obs['dataset_name'].unique())
)

55


In [14]:
for col in ['sampleid', 'age', 'race-stage', 'race', 'group', 'diabetes', 'obesity']:
    dat.obs[col] = dat.obs['dataset_name'].map(meta.set_index('dataset_name')[col])
    
for xcol in ['diabetes','obesity']:
    dat.obs[xcol] = [1 if x=='Y' else 0 for x in dat.obs[xcol]]

xcol = 'group'
dat.obs[xcol] = [0 if x=='HRL' else 1 if x=='DCIS' else 2 for x in dat.obs[xcol]]

xcol = 'race'
dat.obs[xcol] = [0 if x=='NHW' else 1 if x=='NHB' else None for x in dat.obs[xcol]]

dat.obs.columns

Index(['nFeaturess_RNA', 'nFeaturess_RNA_by_counts', 'total_counts',
       'total_counts_mt', 'pct_counts_mt', 'doublets_score', 'nCount_RNA',
       'nFeature_RNA', 'percent.mt', 'predicted.celltype.l1.score',
       'predicted.celltype.l1', 'predicted.celltype.l2.score',
       'predicted.celltype.l2', 'predicted.celltype.l3.score',
       'predicted.celltype.l3', 'mapping.score', 'RNA_snn_res.0.8',
       'seurat_clusters', 'dataset_name', 'RNA_snn_res.0.1', 'S.Score',
       'G2M.Score', 'Phase', 'celltype_level3', 'sampleid', 'age',
       'race-stage', 'race', 'group', 'diabetes', 'obesity'],
      dtype='object')

In [16]:
dat.obs.head(3)

,nFeaturess_RNA,nFeaturess_RNA_by_counts,total_counts,total_counts_mt,pct_counts_mt,doublets_score,nCount_RNA,nFeature_RNA,percent.mt,predicted.celltype.l1.score,...,G2M.Score,Phase,celltype_level3,sampleid,age,race-stage,race,group,diabetes,obesity
Pool137_10_AAACCTGAGACTCGGA-1,1095.0,1095,2669.0,80.0,2.997377,0.039230,2669.0,1095.0,2.997377,0.987477,...,-0.030749,G1,Naive CD4T,1-H,62.3,NHW-HRL,0,0,0,0
Pool137_10_AAACCTGAGATCTGCT-1,2474.0,2472,6188.0,178.0,2.876535,0.021953,6188.0,2472.0,2.876535,1.000000,...,0.004461,S,Nonclassical Monocyte,1-H,62.3,NHW-HRL,0,0,0,0
Pool137_10_AAACCTGAGATTACCC-1,1436.0,1436,3504.0,100.0,2.853881,0.039230,3504.0,1436.0,2.853881,0.996618,...,-0.066766,S,Naive CD8T,1-H,62.3,NHW-HRL,0,0,0,0


# Calc each cell type

In [ ]:
for celltype_major in [
    'B','CD4 T','CD8 T','other T',
    'DC','Mono','NK'
]:
    d_sub = dat[
    dat.obs['predicted.celltype.l1'] == celltype_major
    ]
    
    umap = pd.read_csv(f"data/03_cleandata/ft_{celltype_major}_UmapCoords.csv",
                       index_col=2
                      ).loc[d.obs.index,:]
    
    d_sub.obsm['X_umap'] = umap.values

    sc.pp.neighbors(d_sub)

    covars = ['race', 'age']
    for v in ['obesity', 'diabetes']:
        if v in d_sub.obs and d_sub.obs[v].dropna().nunique() > 1:
            covars.append(v)
            
    samplem = cna.ut.obs_to_sample(d_sub,                           # the dataset
                                   covars,     # the metadata columns to use
                                   'dataset_name')                        # the column of d.obs that contains sample IDs
    
    samplem['batch'] = ['Pool137' if x.startswith('Pool137') else 'Pool138' for x in samplem.index]
    
    
    res = cna.tl.association(
        d_sub,
        samplem.race, 
        'dataset_name',
        covs=samplem[[c for c in covars if c != 'race']],
        batches=samplem.batch,
        key_added='race_coef',
        return_full=True
    )

# B

In [17]:
Counter(dat.obs['predicted.celltype.l1'])

Counter({'CD4 T': 167053,
         'Mono': 108864,
         'CD8 T': 56297,
         'B': 49158,
         'NK': 36671,
         'other T': 8586,
         'DC': 7936})

In [18]:
celltype_major = 'B'

In [19]:
d = dat[dat.obs['predicted.celltype.l1']==celltype_major]

sc.pp.neighbors(d)

In [21]:
pwd

'/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/github'

In [20]:
umap = pd.read_csv(f"data/03_cleandata/ft_{celltype_major}_UmapCoords.csv",
                   index_col=2
                  ).loc[d.obs.index,:]

umap

KeyError: "None of [Index(['Pool137_10_AAACCTGAGGTGATTA-1', 'Pool137_10_AAACGGGGTGCAACGA-1',\n       'Pool137_10_AACACGTGTGTTCTTT-1', 'Pool137_10_AACACGTTCACCTTAT-1',\n       'Pool137_10_AACGTTGAGAGCCCAA-1', 'Pool137_10_AACTTTCCAAAGCAAT-1',\n       'Pool137_10_AAGACCTGTTGTCTTT-1', 'Pool137_10_AAGGTTCAGTTTCCTT-1',\n       'Pool137_10_ACACCCTAGCATCATC-1', 'Pool137_10_ACACCCTAGCCGATTT-1',\n       ...\n       'Pool138_9_TTTACTGTCTAACTCT-1', 'Pool138_9_TTTATGCCAATGACCT-1',\n       'Pool138_9_TTTATGCTCTTGTTTG-1', 'Pool138_9_TTTCCTCAGAGAGCTC-1',\n       'Pool138_9_TTTCCTCGTCAACTGT-1', 'Pool138_9_TTTGCGCAGGGTGTGT-1',\n       'Pool138_9_TTTGGTTCACAGCCCA-1', 'Pool138_9_TTTGGTTTCGTCTGCT-1',\n       'Pool138_9_TTTGTCAAGGAGTAGA-1', 'Pool138_9_TTTGTCACACATAACC-1'],\n      dtype='object', name='umap_1', length=49158)] are in the [index]"

In [ ]:
d.obsm['X_umap'] = umap.values

In [ ]:
samplem = cna.ut.obs_to_sample(d,                           # the dataset
                               ['group','race','age','obesity','diabetes'],     # the metadata columns to use
                               'dataset_name')                        # the column of d.obs that contains sample IDs

samplem['batch'] = ['Pool137' if x.startswith('Pool137') else 'Pool138' for x in samplem.index]

print(Counter(samplem.group))
samplem

In [ ]:
cell_colors = {
    x: palette_all[x] for x in set(d.obs['celltype_level3'])
}


# Build a palette for all cell types: use B cell colors if available, else gray
palette = {}
for ct in d.obs['celltype_level3'].unique():
    if ct in cell_colors:
        palette[ct] = cell_colors[ct]
    else:
        palette[ct] = (0.8, 0.8, 0.8)  # gray for non-B cells

# Plot
fig, ax = plt.subplots(figsize=(3,3), dpi=100)
sc.pl.embedding(
    d,
    basis='X_umap',
    color='celltype_level3',
    palette=palette,
    size=1,
    ax=ax,
    legend_fontsize='large',
    legend_loc='right margin'
)

In [ ]:
# race
p = cna.tl.association(
    d,
    samplem.race, 
    'dataset_name',
    covs=samplem[['age', 'group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='race_coef'
)
d.uns['cnaGlobalPval_race'] = p

# stage
p = cna.tl.association(
    d,
    samplem.group, 
    'dataset_name',
    covs=samplem[['age', 'race','obesity','diabetes']],
    batches=samplem.batch,
    key_added='stage_coef'
)
d.uns['cnaGlobalPval_stage'] = p

# age
p = cna.tl.association(
    d,
    samplem.age, 
    'dataset_name',
    covs=samplem[['race','group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='age_coef'
)
d.uns['cnaGlobalPval_age'] = p

# diabetes
p = cna.tl.association(
    d,
    samplem.diabetes, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='diabetes_coef'
)
d.uns['cnaGlobalPval_diabetes'] = p

# obesity
p = cna.tl.association(
    d,
    samplem.obesity, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='obesity_coef'
)
d.uns['cnaGlobalPval_obesity'] = p

In [ ]:
pkl.dump(d, 
         open(f'data/10_cna/{celltype_major}.pkl', 'wb'))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(figsize=(7, 1.1), ncols=6, nrows=1)
axs = axes.flatten()
plt.subplots_adjust(hspace=0, wspace=0.08)

for idx, covar in enumerate(['race', 'group', 'age', 'diabetes', 'obesity', 'celltype_level3']):
    ax = axs[idx]
    covar_newname = covar if covar != 'group' else 'stage'
    covar_newname = covar if covar != 'group' else 'stage'
    coef_col = f"{covar_newname}_coef"
    fdr_col = f"{covar_newname}_coef_fdr"
    color_col = f"{covar_newname}_coef_plot"

    if covar == 'celltype_level3':
        cell_colors = {x: palette_all[x] for x in set(d.obs['celltype_level3'])}
        palette = {}
        for ct in d.obs['celltype_level3'].unique():
            if ct in cell_colors:
                palette[ct] = cell_colors[ct]
            else:
                palette[ct] = (0.8, 0.8, 0.8)
        sc.pl.umap(
            d,
            color='celltype_level3',
            palette=palette,
            size=.1,
            ax=ax,
            legend_fontsize='large',
            show=False,
            title=''
        )
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(1.05, 0.5),  # right outside the last subplot
            fontsize='large',
            frameon=False
        )
    else:
        p = d.uns[f'cnaGlobalPval_{covar_newname}']
        d.obs[color_col] = np.where(d.obs[fdr_col] < 0.1, d.obs[coef_col], np.nan)   # nan will show as gray in sc.pl.umap
        sc.pl.umap(
            d,
            color=color_col,
            s=.1,
            ax=ax,
            vmin=-1,
            vmax=1,
            cmap='seismic',
            na_color='gray',   # Gray for nan (non-significant)
            colorbar_loc=None,
            title='',
            show=False         # So you can further edit if needed
        )
        
        # Remove colorbar(s) from the figure after plotting
        for extra_ax in ax.figure.axes:
            pos = extra_ax.get_position()
            # Colorbars are usually much narrower than main axes
            if (extra_ax != ax) and (pos.width < 0.1 or pos.height < 0.1):
                ax.figure.delaxes(extra_ax)
                
        ax.text(
            0.01, 0.01,
            f"p = {p:.3f}",
            ha='left', va='bottom',
            fontsize=6,
            transform=ax.transAxes
        )
    ax.set_xlabel('')
    ax.set_ylabel('')

# Reduce space between subplots
# plt.subplots_adjust(wspace=0.1, left=0.08, right=0.95)

# # Place colorbar on the left (tight to first ax)
# cbar_ax = fig.add_axes([0, 0.01, 0.013, 0.6])
# norm = mpl.colors.Normalize(vmin=0, vmax=1)
# sm = mpl.cm.ScalarMappable(norm=norm, cmap='seismic')
# sm.set_array([])
# cbar = fig.colorbar(sm, cax=cbar_ax)

# # Only show 0, 0.5, 1
# cbar.set_ticks([0, 0.5, 1])
# cbar.set_ticklabels(['0', '0.5', '1'])

plt.savefig('plots/Figure4/cna_B.svg')

# CD4 T

In [ ]:
Counter(dat.obs['predicted.celltype.l1'])

In [ ]:
celltype_major = 'CD4 T'

In [ ]:
d = dat[dat.obs['predicted.celltype.l1']==celltype_major]

sc.pp.neighbors(d)

In [ ]:
umap = pd.read_csv(f"data/03_cleandata/ft_{celltype_major}_UmapCoords.csv",
                   index_col=0
                  ).loc[d.obs.index,:]

umap

In [ ]:
d.obsm['X_umap'] = umap.values

In [ ]:
samplem = cna.ut.obs_to_sample(d,                           # the dataset
                               ['group','race','age','obesity','diabetes'],     # the metadata columns to use
                               'dataset_name')                        # the column of d.obs that contains sample IDs

samplem['batch'] = ['Pool137' if x.startswith('Pool137') else 'Pool138' for x in samplem.index]

print(Counter(samplem.group))
samplem

In [ ]:
# race
p = cna.tl.association(
    d,
    samplem.race, 
    'dataset_name',
    covs=samplem[['age', 'group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='race_coef'
)
d.uns['cnaGlobalPval_race'] = p

# stage
p = cna.tl.association(
    d,
    samplem.group, 
    'dataset_name',
    covs=samplem[['age', 'race','obesity','diabetes']],
    batches=samplem.batch,
    key_added='stage_coef'
)
d.uns['cnaGlobalPval_stage'] = p

# age
p = cna.tl.association(
    d,
    samplem.age, 
    'dataset_name',
    covs=samplem[['race','group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='age_coef'
)
d.uns['cnaGlobalPval_age'] = p

# diabetes
p = cna.tl.association(
    d,
    samplem.diabetes, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='diabetes_coef'
)
d.uns['cnaGlobalPval_diabetes'] = p

# obesity
p = cna.tl.association(
    d,
    samplem.obesity, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='obesity_coef'
)
d.uns['cnaGlobalPval_obesity'] = p

In [ ]:
pkl.dump(d, 
         open(f'data/10_cna/{celltype_major}.pkl', 'wb'))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(figsize=(7, 1.1), ncols=6, nrows=1)
axs = axes.flatten()
plt.subplots_adjust(hspace=0, wspace=0.08)

for idx, covar in enumerate(['race', 'group', 'age', 'diabetes', 'obesity', 'celltype_level3']):
    ax = axs[idx]
    covar_newname = covar if covar != 'group' else 'stage'
    covar_newname = covar if covar != 'group' else 'stage'
    coef_col = f"{covar_newname}_coef"
    fdr_col = f"{covar_newname}_coef_fdr"
    color_col = f"{covar_newname}_coef_plot"

    if covar == 'celltype_level3':
        cell_colors = {x: palette_all[x] for x in set(d.obs['celltype_level3'])}
        palette = {}
        for ct in d.obs['celltype_level3'].unique():
            if ct in cell_colors:
                palette[ct] = cell_colors[ct]
            else:
                palette[ct] = (0.8, 0.8, 0.8)
        sc.pl.umap(
            d,
            color='celltype_level3',
            palette=palette,
            size=.1,
            ax=ax,
            legend_fontsize='large',
            show=False,
            title=''
        )
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(1.05, 0.5),  # right outside the last subplot
            fontsize='large',
            frameon=False
        )
    else:
        p = d.uns[f'cnaGlobalPval_{covar_newname}']
        d.obs[color_col] = np.where(d.obs[fdr_col] < 0.1, d.obs[coef_col], np.nan)   # nan will show as gray in sc.pl.umap
        sc.pl.umap(
            d,
            color=color_col,
            s=.1,
            ax=ax,
            vmin=-1,
            vmax=1,
            cmap='seismic',
            na_color='gray',   # Gray for nan (non-significant)
            colorbar_loc=None,
            title='',
            show=False         # So you can further edit if needed
        )
        
        # Remove colorbar(s) from the figure after plotting
        for extra_ax in ax.figure.axes:
            pos = extra_ax.get_position()
            # Colorbars are usually much narrower than main axes
            if (extra_ax != ax) and (pos.width < 0.1 or pos.height < 0.1):
                ax.figure.delaxes(extra_ax)
                
        ax.text(
            0.01, 0.01,
            f"p = {p:.3f}",
            ha='left', va='bottom',
            fontsize=6,
            transform=ax.transAxes
        )
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.savefig(f'plots/Figure5/cna_{celltype_major}.svg')

# CD8 T

In [ ]:
Counter(dat.obs['predicted.celltype.l1'])

In [ ]:
celltype_major = 'CD8 T'

In [ ]:
d = dat[dat.obs['predicted.celltype.l1']==celltype_major]

sc.pp.neighbors(d)

In [ ]:
umap = pd.read_csv(f"data/03_cleandata/ft_{celltype_major}_UmapCoords.csv",
                   index_col=0
                  ).loc[d.obs.index,:]

umap

In [ ]:
d.obsm['X_umap'] = umap.values

In [ ]:
samplem = cna.ut.obs_to_sample(d,                           # the dataset
                               ['group','race','age','obesity','diabetes'],     # the metadata columns to use
                               'dataset_name')                        # the column of d.obs that contains sample IDs

samplem['batch'] = ['Pool137' if x.startswith('Pool137') else 'Pool138' for x in samplem.index]

print(Counter(samplem.group))
samplem

In [ ]:
# race
p = cna.tl.association(
    d,
    samplem.race, 
    'dataset_name',
    covs=samplem[['age', 'group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='race_coef'
)
d.uns['cnaGlobalPval_race'] = p

# stage
p = cna.tl.association(
    d,
    samplem.group, 
    'dataset_name',
    covs=samplem[['age', 'race','obesity','diabetes']],
    batches=samplem.batch,
    key_added='stage_coef'
)
d.uns['cnaGlobalPval_stage'] = p

# age
p = cna.tl.association(
    d,
    samplem.age, 
    'dataset_name',
    covs=samplem[['race','group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='age_coef'
)
d.uns['cnaGlobalPval_age'] = p

# diabetes
p = cna.tl.association(
    d,
    samplem.diabetes, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='diabetes_coef'
)
d.uns['cnaGlobalPval_diabetes'] = p

# obesity
p = cna.tl.association(
    d,
    samplem.obesity, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='obesity_coef'
)
d.uns['cnaGlobalPval_obesity'] = p

In [ ]:
pkl.dump(d, 
         open(f'data/10_cna/{celltype_major}.pkl', 'wb'))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(figsize=(7, 1.1), ncols=6, nrows=1)
axs = axes.flatten()
plt.subplots_adjust(hspace=0, wspace=0.08)

for idx, covar in enumerate(['race', 'group', 'age', 'diabetes', 'obesity', 'celltype_level3']):
    ax = axs[idx]
    covar_newname = covar if covar != 'group' else 'stage'
    covar_newname = covar if covar != 'group' else 'stage'
    coef_col = f"{covar_newname}_coef"
    fdr_col = f"{covar_newname}_coef_fdr"
    color_col = f"{covar_newname}_coef_plot"

    if covar == 'celltype_level3':
        cell_colors = {x: palette_all[x] for x in set(d.obs['celltype_level3'])}
        palette = {}
        for ct in d.obs['celltype_level3'].unique():
            if ct in cell_colors:
                palette[ct] = cell_colors[ct]
            else:
                palette[ct] = (0.8, 0.8, 0.8)
        sc.pl.umap(
            d,
            color='celltype_level3',
            palette=palette,
            size=.1,
            ax=ax,
            legend_fontsize='large',
            show=False,
            title=''
        )
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(1.05, 0.5),  # right outside the last subplot
            fontsize='large',
            frameon=False
        )
    else:
        p = d.uns[f'cnaGlobalPval_{covar_newname}']
        d.obs[color_col] = np.where(d.obs[fdr_col] < 0.1, d.obs[coef_col], np.nan)   # nan will show as gray in sc.pl.umap
        sc.pl.umap(
            d,
            color=color_col,
            s=.1,
            ax=ax,
            vmin=-1,
            vmax=1,
            cmap='seismic',
            na_color='gray',   # Gray for nan (non-significant)
            colorbar_loc=None,
            title='',
            show=False         # So you can further edit if needed
        )
        
        # Remove colorbar(s) from the figure after plotting
        for extra_ax in ax.figure.axes:
            pos = extra_ax.get_position()
            # Colorbars are usually much narrower than main axes
            if (extra_ax != ax) and (pos.width < 0.1 or pos.height < 0.1):
                ax.figure.delaxes(extra_ax)
                
        ax.text(
            1, 0.01,
            f"p = {p:.3f}",
            ha='right', va='bottom',
            fontsize=6,
            transform=ax.transAxes
        )
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.savefig(f'plots/Figure4/cna_{celltype_major}.svg')

# other T

In [ ]:
Counter(dat.obs['predicted.celltype.l1'])

In [ ]:
celltype_major = 'other T'

In [ ]:
d = dat[dat.obs['predicted.celltype.l1']==celltype_major]

sc.pp.neighbors(d)

In [ ]:
umap = pd.read_csv(f"data/03_cleandata/ft_{celltype_major}_UmapCoords.csv",
                   index_col=0
                  ).loc[d.obs.index,:]

umap

In [ ]:
d.obsm['X_umap'] = umap.values

In [ ]:
samplem = cna.ut.obs_to_sample(d,                           # the dataset
                               ['group','race','age','obesity','diabetes'],     # the metadata columns to use
                               'dataset_name')                        # the column of d.obs that contains sample IDs

samplem['batch'] = ['Pool137' if x.startswith('Pool137') else 'Pool138' for x in samplem.index]

print(Counter(samplem.group))
samplem

In [ ]:
# race
p = cna.tl.association(
    d,
    samplem.race, 
    'dataset_name',
    covs=samplem[['age', 'group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='race_coef'
)
d.uns['cnaGlobalPval_race'] = p

# stage
p = cna.tl.association(
    d,
    samplem.group, 
    'dataset_name',
    covs=samplem[['age', 'race','obesity','diabetes']],
    batches=samplem.batch,
    key_added='stage_coef'
)
d.uns['cnaGlobalPval_stage'] = p

# age
p = cna.tl.association(
    d,
    samplem.age, 
    'dataset_name',
    covs=samplem[['race','group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='age_coef'
)
d.uns['cnaGlobalPval_age'] = p

# diabetes
p = cna.tl.association(
    d,
    samplem.diabetes, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='diabetes_coef'
)
d.uns['cnaGlobalPval_diabetes'] = p

# obesity
p = cna.tl.association(
    d,
    samplem.obesity, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='obesity_coef'
)
d.uns['cnaGlobalPval_obesity'] = p

In [ ]:
pkl.dump(d, 
         open(f'data/10_cna/{celltype_major}.pkl', 'wb'))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(figsize=(7, 1.1), ncols=6, nrows=1)
axs = axes.flatten()
plt.subplots_adjust(hspace=0, wspace=0.08)

for idx, covar in enumerate(['race', 'group', 'age', 'diabetes', 'obesity', 'celltype_level3']):
    ax = axs[idx]
    covar_newname = covar if covar != 'group' else 'stage'
    covar_newname = covar if covar != 'group' else 'stage'
    coef_col = f"{covar_newname}_coef"
    fdr_col = f"{covar_newname}_coef_fdr"
    color_col = f"{covar_newname}_coef_plot"

    if covar == 'celltype_level3':
        cell_colors = {x: palette_all[x] for x in set(d.obs['celltype_level3'])}
        palette = {}
        for ct in d.obs['celltype_level3'].unique():
            if ct in cell_colors:
                palette[ct] = cell_colors[ct]
            else:
                palette[ct] = (0.8, 0.8, 0.8)
        sc.pl.umap(
            d,
            color='celltype_level3',
            palette=palette,
            size=.1,
            ax=ax,
            legend_fontsize='large',
            show=False,
            title=''
        )
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(1.05, 0.5),  # right outside the last subplot
            fontsize='large',
            frameon=False
        )
    else:
        p = d.uns[f'cnaGlobalPval_{covar_newname}']
        d.obs[color_col] = np.where(d.obs[fdr_col] < 0.1, d.obs[coef_col], np.nan)   # nan will show as gray in sc.pl.umap
        sc.pl.umap(
            d,
            color=color_col,
            s=.1,
            ax=ax,
            vmin=-1,
            vmax=1,
            cmap='seismic',
            na_color='gray',   # Gray for nan (non-significant)
            colorbar_loc=None,
            title='',
            show=False         # So you can further edit if needed
        )
        
        # Remove colorbar(s) from the figure after plotting
        for extra_ax in ax.figure.axes:
            pos = extra_ax.get_position()
            # Colorbars are usually much narrower than main axes
            if (extra_ax != ax) and (pos.width < 0.1 or pos.height < 0.1):
                ax.figure.delaxes(extra_ax)
                
        ax.text(
            1, 0.01,
            f"p = {p:.3f}",
            ha='right', va='bottom',
            fontsize=6,
            transform=ax.transAxes
        )
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.savefig(f'plots/Figure4/cna_{celltype_major}.svg')

# Mono

In [ ]:
Counter(dat.obs['predicted.celltype.l1'])

In [ ]:
celltype_major = 'Mono'

In [ ]:
d = dat[dat.obs['predicted.celltype.l1']==celltype_major]

sc.pp.neighbors(d)

In [ ]:
umap = pd.read_csv(f"data/03_cleandata/ft_{celltype_major}_UmapCoords.csv",
                   index_col=0
                  ).loc[d.obs.index,:]

umap

In [ ]:
d.obsm['X_umap'] = umap.values

In [ ]:
samplem = cna.ut.obs_to_sample(d,                           # the dataset
                               ['group','race','age','obesity','diabetes'],     # the metadata columns to use
                               'dataset_name')                        # the column of d.obs that contains sample IDs

samplem['batch'] = ['Pool137' if x.startswith('Pool137') else 'Pool138' for x in samplem.index]

print(Counter(samplem.group))
samplem

In [ ]:
# race
p = cna.tl.association(
    d,
    samplem.race, 
    'dataset_name',
    covs=samplem[['age', 'group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='race_coef',
    return_full=True
)
d.uns['cnaGlobalPval_race'] = p

# stage
p = cna.tl.association(
    d,
    samplem.group, 
    'dataset_name',
    covs=samplem[['age', 'race','obesity','diabetes']],
    batches=samplem.batch,
    key_added='stage_coef'
)
d.uns['cnaGlobalPval_stage'] = p

# age
p = cna.tl.association(
    d,
    samplem.age, 
    'dataset_name',
    covs=samplem[['race','group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='age_coef'
)
d.uns['cnaGlobalPval_age'] = p

# diabetes
p = cna.tl.association(
    d,
    samplem.diabetes, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='diabetes_coef'
)
d.uns['cnaGlobalPval_diabetes'] = p

# obesity
p = cna.tl.association(
    d,
    samplem.obesity, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='obesity_coef'
)
d.uns['cnaGlobalPval_obesity'] = p

In [ ]:
pkl.dump(d, 
         open(f'data/10_cna/{celltype_major}.pkl', 'wb'))


In [ ]:
celltype_major = 'Mono'

with open(f'data/10_cna/{celltype_major}.pkl','rb') as f:
    d = pkl.load(f)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(figsize=(7, 1.1), ncols=6, nrows=1)
axs = axes.flatten()
plt.subplots_adjust(hspace=0, wspace=0.08)

for idx, covar in enumerate(['race', 'group', 'age', 'diabetes', 'obesity', 'celltype_level3']):
    ax = axs[idx]
    covar_newname = covar if covar != 'group' else 'stage'
    covar_newname = covar if covar != 'group' else 'stage'
    coef_col = f"{covar_newname}_coef"
    fdr_col = f"{covar_newname}_coef_fdr"
    color_col = f"{covar_newname}_coef_plot"

    if covar == 'celltype_level3':
        cell_colors = {x: palette_all[x] for x in set(d.obs['celltype_level3'])}
        palette = {}
        for ct in d.obs['celltype_level3'].unique():
            if ct in cell_colors:
                palette[ct] = cell_colors[ct]
            else:
                palette[ct] = (0.8, 0.8, 0.8)
        sc.pl.umap(
            d,
            color='celltype_level3',
            palette=palette,
            size=.1,
            ax=ax,
            legend_fontsize='large',
            show=False,
            title=''
        )
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(1.05, 0.5),  # right outside the last subplot
            fontsize='large',
            frameon=False
        )
    else:
        p = d.uns[f'cnaGlobalPval_{covar_newname}']
        d.obs[color_col] = np.where(d.obs[fdr_col] < 0.1, d.obs[coef_col], np.nan)   # nan will show as gray in sc.pl.umap
        sc.pl.umap(
            d,
            color=color_col,
            s=.1,
            ax=ax,
            vmin=-1,
            vmax=1,
            cmap='seismic',
            na_color='gray',   # Gray for nan (non-significant)
            colorbar_loc=None,
            title='',
            show=False         # So you can further edit if needed
        )
        
        # Remove colorbar(s) from the figure after plotting
        for extra_ax in ax.figure.axes:
            pos = extra_ax.get_position()
            # Colorbars are usually much narrower than main axes
            if (extra_ax != ax) and (pos.width < 0.1 or pos.height < 0.1):
                ax.figure.delaxes(extra_ax)
                
        ax.text(
            0.01, 0.01,
            f"p = {p:.3f}",
            ha='left', va='bottom',
            fontsize=6,
            transform=ax.transAxes
        )
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.savefig(f'plots/Figure4/cna_{celltype_major}.svg')

# DC

In [ ]:
Counter(dat.obs['predicted.celltype.l1'])

In [ ]:
celltype_major = 'DC'

In [ ]:
d = dat[dat.obs['predicted.celltype.l1']==celltype_major]

sc.pp.neighbors(d)

In [ ]:
umap = pd.read_csv(f"data/03_cleandata/ft_{celltype_major}_UmapCoords.csv",
                   index_col=0
                  ).loc[d.obs.index,:]

umap

In [ ]:
d.obsm['X_umap'] = umap.values

In [ ]:
samplem = cna.ut.obs_to_sample(d,                           # the dataset
                               ['group','race','age','obesity','diabetes'],     # the metadata columns to use
                               'dataset_name')                        # the column of d.obs that contains sample IDs

samplem['batch'] = ['Pool137' if x.startswith('Pool137') else 'Pool138' for x in samplem.index]

print(Counter(samplem.group))
samplem

In [ ]:
# race
p = cna.tl.association(
    d,
    samplem.race, 
    'dataset_name',
    covs=samplem[['age', 'group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='race_coef'
)
d.uns['cnaGlobalPval_race'] = p

# stage
p = cna.tl.association(
    d,
    samplem.group, 
    'dataset_name',
    covs=samplem[['age', 'race','obesity','diabetes']],
    batches=samplem.batch,
    key_added='stage_coef'
)
d.uns['cnaGlobalPval_stage'] = p

# age
p = cna.tl.association(
    d,
    samplem.age, 
    'dataset_name',
    covs=samplem[['race','group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='age_coef'
)
d.uns['cnaGlobalPval_age'] = p

# diabetes
p = cna.tl.association(
    d,
    samplem.diabetes, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='diabetes_coef'
)
d.uns['cnaGlobalPval_diabetes'] = p

# obesity
p = cna.tl.association(
    d,
    samplem.obesity, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='obesity_coef'
)
d.uns['cnaGlobalPval_obesity'] = p

In [ ]:
pkl.dump(d, 
         open(f'data/10_cna/{celltype_major}.pkl', 'wb'))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(figsize=(7, 1.1), ncols=6, nrows=1)
axs = axes.flatten()
plt.subplots_adjust(hspace=0, wspace=0.08)

for idx, covar in enumerate(['race', 'group', 'age', 'diabetes', 'obesity', 'celltype_level3']):
    ax = axs[idx]
    covar_newname = covar if covar != 'group' else 'stage'
    covar_newname = covar if covar != 'group' else 'stage'
    coef_col = f"{covar_newname}_coef"
    fdr_col = f"{covar_newname}_coef_fdr"
    color_col = f"{covar_newname}_coef_plot"

    if covar == 'celltype_level3':
        cell_colors = {x: palette_all[x] for x in set(d.obs['celltype_level3'])}
        palette = {}
        for ct in d.obs['celltype_level3'].unique():
            if ct in cell_colors:
                palette[ct] = cell_colors[ct]
            else:
                palette[ct] = (0.8, 0.8, 0.8)
        sc.pl.umap(
            d,
            color='celltype_level3',
            palette=palette,
            size=.1,
            ax=ax,
            legend_fontsize='large',
            show=False,
            title=''
        )
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(1.05, 0.5),  # right outside the last subplot
            fontsize='large',
            frameon=False
        )
    else:
        p = d.uns[f'cnaGlobalPval_{covar_newname}']
        d.obs[color_col] = np.where(d.obs[fdr_col] < 0.1, d.obs[coef_col], np.nan)   # nan will show as gray in sc.pl.umap
        sc.pl.umap(
            d,
            color=color_col,
            s=.1,
            ax=ax,
            vmin=-1,
            vmax=1,
            cmap='seismic',
            na_color='gray',   # Gray for nan (non-significant)
            colorbar_loc=None,
            title='',
            show=False         # So you can further edit if needed
        )
        
        # Remove colorbar(s) from the figure after plotting
        for extra_ax in ax.figure.axes:
            pos = extra_ax.get_position()
            # Colorbars are usually much narrower than main axes
            if (extra_ax != ax) and (pos.width < 0.1 or pos.height < 0.1):
                ax.figure.delaxes(extra_ax)
                
        ax.text(
            1, 0.01,
            f"p = {p:.3f}",
            ha='right', va='bottom',
            fontsize=6,
            transform=ax.transAxes
        )
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.savefig(f'plots/Figure4/cna_{celltype_major}.svg')

# NK

In [ ]:
Counter(dat.obs['predicted.celltype.l1'])

In [ ]:
celltype_major = 'NK'

In [ ]:
d = dat[dat.obs['predicted.celltype.l1']==celltype_major]

sc.pp.neighbors(d)

In [ ]:
umap = pd.read_csv(f"data/03_cleandata/ft_{celltype_major}_UmapCoords.csv",
                   index_col=0
                  ).loc[d.obs.index,:]

umap

In [ ]:
d.obsm['X_umap'] = umap.values

In [ ]:
samplem = cna.ut.obs_to_sample(d,                           # the dataset
                               ['group','race','age','obesity','diabetes'],     # the metadata columns to use
                               'dataset_name')                        # the column of d.obs that contains sample IDs

samplem['batch'] = ['Pool137' if x.startswith('Pool137') else 'Pool138' for x in samplem.index]

print(Counter(samplem.group))
samplem

In [ ]:
# race
p = cna.tl.association(
    d,
    samplem.race, 
    'dataset_name',
    covs=samplem[['age', 'group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='race_coef'
)
d.uns['cnaGlobalPval_race'] = p

# stage
p = cna.tl.association(
    d,
    samplem.group, 
    'dataset_name',
    covs=samplem[['age', 'race','obesity','diabetes']],
    batches=samplem.batch,
    key_added='stage_coef'
)
d.uns['cnaGlobalPval_stage'] = p

# age
p = cna.tl.association(
    d,
    samplem.age, 
    'dataset_name',
    covs=samplem[['race','group','obesity','diabetes']],
    batches=samplem.batch,
    key_added='age_coef'
)
d.uns['cnaGlobalPval_age'] = p

# diabetes
p = cna.tl.association(
    d,
    samplem.diabetes, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='diabetes_coef'
)
d.uns['cnaGlobalPval_diabetes'] = p

# obesity
p = cna.tl.association(
    d,
    samplem.obesity, 
    'dataset_name',
    covs=samplem[['age','race','group','obesity']],
    batches=samplem.batch,
    key_added='obesity_coef'
)
d.uns['cnaGlobalPval_obesity'] = p

In [ ]:
pkl.dump(d, 
         open(f'data/10_cna/{celltype_major}.pkl', 'wb'))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

fig, axes = plt.subplots(figsize=(7, 1.1), ncols=6, nrows=1)
axs = axes.flatten()
plt.subplots_adjust(hspace=0, wspace=0.08)

for idx, covar in enumerate(['race', 'group', 'age', 'diabetes', 'obesity', 'celltype_level3']):
    ax = axs[idx]
    covar_newname = covar if covar != 'group' else 'stage'
    covar_newname = covar if covar != 'group' else 'stage'
    coef_col = f"{covar_newname}_coef"
    fdr_col = f"{covar_newname}_coef_fdr"
    color_col = f"{covar_newname}_coef_plot"

    if covar == 'celltype_level3':
        cell_colors = {x: palette_all[x] for x in set(d.obs['celltype_level3'])}
        palette = {}
        for ct in d.obs['celltype_level3'].unique():
            if ct in cell_colors:
                palette[ct] = cell_colors[ct]
            else:
                palette[ct] = (0.8, 0.8, 0.8)
        sc.pl.umap(
            d,
            color='celltype_level3',
            palette=palette,
            size=.1,
            ax=ax,
            legend_fontsize='large',
            show=False,
            title=''
        )
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(1.05, 0.5),  # right outside the last subplot
            fontsize='large',
            frameon=False
        )
    else:
        p = d.uns[f'cnaGlobalPval_{covar_newname}']
        d.obs[color_col] = np.where(d.obs[fdr_col] < 0.1, d.obs[coef_col], np.nan)   # nan will show as gray in sc.pl.umap
        sc.pl.umap(
            d,
            color=color_col,
            s=.1,
            ax=ax,
            vmin=-1,
            vmax=1,
            cmap='seismic',
            na_color='gray',   # Gray for nan (non-significant)
            colorbar_loc=None,
            title='',
            show=False         # So you can further edit if needed
        )
        
        # Remove colorbar(s) from the figure after plotting
        for extra_ax in ax.figure.axes:
            pos = extra_ax.get_position()
            # Colorbars are usually much narrower than main axes
            if (extra_ax != ax) and (pos.width < 0.1 or pos.height < 0.1):
                ax.figure.delaxes(extra_ax)
                
        ax.text(
            0.01, 0.01,
            f"p = {p:.3f}",
            ha='left', va='bottom',
            fontsize=6,
            transform=ax.transAxes
        )
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.savefig(f'plots/Figure4/cna_{celltype_major}.svg')

In [ ]:
d.obs.columns

# Summarize - Neighborhood coef & FDR

In [ ]:
celltypes_major = [
    'B',  'CD4 T','CD8 T','other T',  'DC','Mono','NK'
]

colnames_sel = [
    'dataset_name','Phase','sampleid', 
    'age', 'race-stage', 'race', 'group', 'diabetes', 'obesity',
    'celltype_level3','predicted.celltype.l1',
    
    'race_coef', 'race_coef_fdr', 
    'stage_coef', 'stage_coef_fdr', 
    'age_coef', 'age_coef_fdr', 
    'diabetes_coef', 'diabetes_coef_fdr', 
    'obesity_coef', 'obesity_coef_fdr'
]

for idx, celltype_major in enumerate(celltypes_major):

    with open(f'data/10_cna/{celltype_major}.pkl','rb') as f:
        
        d = pkl.load(f)
        tmp = d.obs.loc[:,colnames_sel].copy()
        
        tmp['UMAP1'] = d.obsm['X_umap'][:,0]
        tmp['UMAP2'] = d.obsm['X_umap'][:,1]

        out = tmp if idx==0 else pd.concat((out, tmp))
    
out

In [ ]:
out.to_csv('data/Figure5/cna_coef.csv')

In [ ]:
out = pd.read_csv('data/Figure5/cna_coef.csv', index_col=0)
out

In [ ]:
covars = ['race', 'stage', 'age', 'diabetes', 'obesity']
celltypes = out['celltype_level3'].unique()

records = []
for covar in covars:
    fdr_col = f'{covar}_coef_fdr'
    coef_col = f'{covar}_coef'
    for celltype in celltypes:
        subset = out[out['celltype_level3'] == celltype]
        sig = subset[subset[fdr_col] < 0.1]
        count_lt_0_1 = len(sig)
        count_ge_0_1 = (subset[fdr_col] >= 0.1).sum()
        # Positive and negative (within significant)
        count_pos = (sig[coef_col] > 0).sum()
        count_neg = (sig[coef_col] < 0).sum()
        pct_pos = (count_pos / (count_lt_0_1 + count_ge_0_1)) * 100 if count_lt_0_1 > 0 else 0
        pct_neg = (count_neg / (count_lt_0_1 + count_ge_0_1)) * 100 if count_lt_0_1 > 0 else 0
        
        records.append({
            'covar': covar,
            'celltype': celltype,
            'count_fdr_lt_0.1': count_lt_0_1,
            'count_fdr_ge_0.1': count_ge_0_1,
            'pct_sigbyFDR0.1': count_lt_0_1 / (count_lt_0_1 + count_ge_0_1) * 100 if (count_lt_0_1 + count_ge_0_1) > 0 else 0,
            'pct_pos_sig': pct_pos,
            'pct_neg_sig': pct_neg
        })

df_counts = pd.DataFrame(records)
df_counts

In [ ]:
df_counts.to_csv('data/Figure5/cna_counts.csv')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

celltype_to_major = {}
for group, members in cells_group.items():
    for cell in members:
        celltype_to_major[cell] = group
        
# Covariate group base color
covar_color_pos = {
    'race':  '#64a3e0',    # warm blue (positive)
    'stage': '#f29d5c',    # warm orange
    'age':   '#66c2a5',    # warm teal
}
covar_color_neg = {
    'race':  '#bfd6ef',    # cold blue (negative)
    'stage': '#fbe3d2',    # cold orange
    'age':   '#d2f2e2',    # cold teal
}
bar_alpha = 0.8

x_labels = []
bar_pos = []
bar_neg = []
bar_color_pos = []
bar_color_neg = []
section_starts = []

for covar in ['race', 'stage', 'age']:
    subset = df_counts[(df_counts['covar'] == covar) & (df_counts['count_fdr_lt_0.1'] > 0)]
    if len(subset) > 0:
        section_starts.append(len(x_labels))
        for _, row in subset.iterrows():
            x_labels.append(row['celltype'])
            bar_pos.append(row['pct_pos_sig'])
            bar_neg.append(-row['pct_neg_sig'])   # negative, so bar goes down
            bar_color_pos.append(covar_color_pos[covar])
            bar_color_neg.append(covar_color_neg[covar])

plt.figure(figsize=(4, 1.5))

# Plot positive bars (upwards)
plt.bar(
    range(len(x_labels)),
    bar_pos,
    color=bar_color_pos,
    alpha=bar_alpha,
    width=0.8
)
# Plot negative bars (downwards)
plt.bar(
    range(len(x_labels)),
    bar_neg,
    color=bar_color_neg,
    alpha=bar_alpha,
    width=0.8
)

plt.ylabel('Significant cells%', fontsize=7)
plt.title('')
plt.xticks(
    range(len(x_labels)),
    x_labels,
    rotation=30, ha='right', fontsize=5
)
plt.xlabel('')
plt.xlim(-0.5, len(x_labels)-0.5)
plt.ylim(-30,30)

# Custom y-tick labels: e.g. "20 pos", "10 pos", 0, "10 neg", "20 neg"
yticks = [-30,-20,0,20,30]  # -20, -10, 0, 10, 20
yticklabels = ['Neg','-20','0','20','Pos']


plt.yticks(yticks, yticklabels, fontsize=7)

#


# Color x-tick labels by major cell type
for idx, label in enumerate(plt.gca().get_xticklabels()):
    celltype = x_labels[idx]
    major = celltype_to_major.get(celltype, None)
    if major:
        label.set_color(major_celltype_palette[major])
    else:
        label.set_color('black')

# Annotate boundaries between sections
for start in section_starts[1:]:
    plt.axvline(start-0.5, color='gray', linestyle='--', lw=0.7)

plt.axhline(0, color='black', linewidth=0.7)

plt.tight_layout()
plt.savefig('plots/Figure4/cna_barplot_bycovariateFDRcellcounts_posneg.svg')




# (Optional) Build and save a legend separately
legend_handles = [
    Patch(color=covar_color_pos['race'], label='Race, Positive', alpha=bar_alpha),
    Patch(color=covar_color_neg['race'], label='Race, Negative', alpha=bar_alpha),
    Patch(color=covar_color_pos['stage'], label='Stage, Positive', alpha=bar_alpha),
    Patch(color=covar_color_neg['stage'], label='Stage, Negative', alpha=bar_alpha),
    Patch(color=covar_color_pos['age'], label='Age, Positive', alpha=bar_alpha),
    Patch(color=covar_color_neg['age'], label='Age, Negative', alpha=bar_alpha),
]

fig_leg, ax_leg = plt.subplots(figsize=(4.2, 0.45))
ax_leg.axis('off')
leg = ax_leg.legend(
    handles=legend_handles,
    loc='center',
    frameon=False,
    ncol=3,
    fontsize=8,
    handlelength=1.8,
    columnspacing=1.2
)
fig_leg.savefig('plots/Figure5/cna_barplot_posneg_legend.svg', bbox_inches='tight', transparent=True)
plt.close(fig_leg)

# Summarize - Global P

In [ ]:
celltypes_major = [
    'B',  'CD4 T','CD8 T','other T',  'DC','Mono','NK'
]

colnames_sel = [
    'cnaGlobalPval_race',
    'cnaGlobalPval_stage',
    'cnaGlobalPval_age',
    'cnaGlobalPval_diabetes',
    'cnaGlobalPval_obesity'    
]

pvals_master = []

for idx, celltype_major in enumerate(celltypes_major):

    with open(f'data/10_cna/{celltype_major}.pkl','rb') as f:
        
        d = pkl.load(f)
        pvals = []
        for cname in colnames_sel:
            pvals.append(d.uns[cname])
            
        pvals_master.append(pvals)


In [ ]:
pvals_df = pd.DataFrame(np.array(pvals_master))

pvals_df.columns = colnames_sel
pvals_df.index = celltypes_major

pvals_df

In [ ]:
pvals_df.to_csv('data/Figure5/cna_globalP.csv')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
# Define color palettes
group_palette = {
    'race':     '#64a3e0',
    'stage':    '#f29d5c',
    'age':      '#66c2a5',
    'diabetes': '#F472B6',
    'obesity':  '#F59E42',
}
major_celltype_palette = {
    'B':    '#3072C4',
    'T':    '#EF763F',
    'DC':   '#35A293',
    'Mono': '#B3813E',
    'NK':   '#7D4C99',
}

# Helper function to determine major cell type group
def get_major_celltype(celltype):
    if 'CD4' in celltype or 'CD8' in celltype or 'T' in celltype:
        return 'T'
    for key in major_celltype_palette:
        if key in celltype:
            return key
    return 'other'

# Variables to plot, in order
variables = [
    ('cnaGlobalPval_race', 'race'),
    ('cnaGlobalPval_stage', 'stage'),
    ('cnaGlobalPval_age', 'age'),
    ('cnaGlobalPval_diabetes', 'diabetes'),
    ('cnaGlobalPval_obesity', 'obesity')
]

celltypes = pvals_df.index.tolist()
n_celltypes = len(celltypes)
n_groups = len(variables)

# Build x and y for plotting
x_vals = []
y_vals = []
colors = []
tick_labels = []
tick_colors = []

for group_idx, (colname, group_label) in enumerate(variables):
    for i, ct in enumerate(celltypes):
        x = group_idx * n_celltypes + i
        y = -np.log10(pvals_df.loc[ct, colname])
        color = group_palette[group_label]
        tick_color = major_celltype_palette.get(get_major_celltype(ct), 'black')

        x_vals.append(x)
        y_vals.append(y)
        colors.append(color)
        tick_labels.append(ct)
        tick_colors.append(tick_color)

# Plotting
plt.figure(figsize=(3.2, 1.5))

# Draw string-dots
for x, y, c in zip(x_vals, y_vals, colors):
    plt.plot([x, x], [0, y], color=c, linewidth=0.6)
    plt.scatter(x, y, color=c, s=8)
plt.ylim(bottom=0)

# Horizontal line for significance cutoff
plt.axhline(-np.log10(0.05), linestyle='--', color='maroon', linewidth=0.8)

# X-axis
plt.xticks(range(len(x_vals)), tick_labels, rotation=90, ha='center', fontsize=6)
ax = plt.gca()
for label, color in zip(ax.get_xticklabels(), tick_colors * n_groups):
    label.set_color(color)

# Labels
plt.ylabel(r'$-\log_{10}(p)$', fontsize=7)
plt.title('CNA global association', fontsize=8)

# Optional: add color legend (optional)
legend_elements = [
    Line2D(
        [0], [0],
        marker='o',
        color='none',  # no line
        markerfacecolor=group_palette[g],
        markeredgewidth=0,  # no border/stroke
        markersize=5,
        label=g.capitalize()
    )
    for _, g in variables
]

plt.legend(
    handles=legend_elements,
    bbox_to_anchor=(1, 1),  # right-aligned, top
    loc='upper left',
    fontsize=6,
    frameon=False,
    borderaxespad=0  # removes padding between legend and axes
)


plt.tight_layout()
plt.savefig('plots/Figure5/CNA_globalP.svg')

# Overlap, global & local p

In [ ]:
from matplotlib_venn import venn2, venn3

In [ ]:
local_p = pd.read_csv('data/Figure5/cna_counts.csv', index_col=0)
local_p

In [ ]:
global_p = pd.read_csv('data/Figure5/cna_globalP.csv', index_col=0)
global_p

In [ ]:
# Wide → Long
global_p_long = (
    global_p
    .reset_index()
    .melt(
        id_vars='index',
        value_vars=['cnaGlobalPval_race', 'cnaGlobalPval_stage',
                    'cnaGlobalPval_age', 'cnaGlobalPval_diabetes',
                    'cnaGlobalPval_obesity'],
        var_name='covar',
        value_name='Pval'
    )
)

# Extract covariate name (remove prefix "cnaGlobalPval_")
global_p_long['covar'] = global_p_long['covar'].str.replace('cnaGlobalPval_', '', regex=False)

# Rename cell type column
global_p_long = global_p_long.rename(columns={'index': 'celltype'})
global_p_long

In [ ]:
global_p_long['covar'].unique()

In [ ]:
cells_group = {
'B' :    ['Naive B','Mature B','Memory B','Plasmablast'],
'CD4 T' :   ['Naive CD4T','Central Memory CD4T','Effector Memory CD4T','Cytotoxic CD4T','Treg'],
'CD8 T': ['Naive CD8T','Central Memory CD8T','Effector Memory CD8T'],
'Other T': ['Double Negative T','γδ T','MAIT'],
'DC' :    ['cDC1','cDC2','pDC'],
'Mono' :    ['Classical Monocyte','Intermediate Monocyte','Nonclassical Monocyte','Cytotoxic Monocyte'],
'NK' :    ['CD56-dim NK','CD56-bright NK','Proliferating NK']
}

cell_to_group = {
    cell: group
    for group, cells in cells_group.items()
    for cell in cells
}

In [ ]:
def plot_venn_outside_labels(ax, s1, s2, 
                             set_labels=('Global','Local'),
                             show_legend=False
                            ):

    from venn import generate_petal_labels, draw_venn
    import matplotlib.colors as mcolors
    
    # Prepare dict for venn
    sets_for_plot = {
        set_labels[0]: s1 if len(s1) else {'__dummy_g__'},
        set_labels[1]: s2 if len(s2) else {'__dummy_l__'},
    }
    # sets_for_plot = {
    #     set_labels[0]: s1 ,
    #     set_labels[1]: s2 ,
    # }
    # Plot venn
    # venn(sets_for_plot, ax=ax, fontsize=8)

    # Compute counts
    only1 = len(s1 - s2)
    only2 = len(s2 - s1)
    both  = len(s1 & s2)

    print('Only1:',(s1 - s2))
    print('Only2:',(s2 - s1))
    print('Both:',s1 & s2)
    print('\n\n')

    petal_labels = {
    '10': only1,
    '01': only2,
    '11': both
    }

    venn_colors = [
            mcolors.to_rgba('purple', alpha=0.5),
            mcolors.to_rgba('yellow', alpha=0.5)
        ]
    
    draw_venn(
        petal_labels=petal_labels,
        dataset_labels=sets_for_plot.keys(),
        hint_hidden=False,
        colors=venn_colors,
        figsize=(2,2),
        fontsize=7,
        legend_loc="lower right",
        ax=ax  # or pass a specific matplotlib axis
    )

    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.get_legend().remove()

    if show_legend:
        handles = [
            Patch(facecolor=venn_colors[0], edgecolor="black", label=set_labels[0]),
            Patch(facecolor=venn_colors[1], edgecolor="black", label=set_labels[1])
        ]
        ax.legend(handles=handles, frameon=False, fontsize=7, loc="center right",
                 bbox_to_anchor=(1.3, 0.5))
    else:
        if ax.get_legend():
            ax.get_legend().remove()



In [ ]:
fig, axs = plt.subplots(figsize=(7, 1.5), ncols=5)

plt.subplots_adjust(wspace=.01)

for idx, covar in enumerate(['race', 'stage', 'age', 'diabetes', 'obesity']): # 

    ax = axs.flatten()[idx]

    sub = global_p_long[global_p_long['covar']==covar]
    s1 = set(sub[sub['Pval']<0.05]['celltype'])

    sub = local_p[local_p['covar']==covar]
    s2 = set(sub[(sub['pct_pos_sig']>0) | (sub['pct_neg_sig']>0)]['celltype'])
    s2 = {cell_to_group.get(cell, 'Other') for cell in s2}

    print(covar)
    lgd = True if covar == 'obesity' else False
    plot_venn_outside_labels(ax, s1, s2, show_legend=lgd)
    
    ax.set_title(f"{covar.capitalize()}", fontsize=7, pad=-5)
    ax.title.set_position([0.5, -1]) 

plt.savefig('plots/Figure5/Venn_byCovars.svg', bbox_inches='tight')

# Replot, coefs

In [ ]:
for celltype_major in ['B','CD4 T','CD8 T','other T','DC','Mono','NK']:

    d = pkl.load(
             open(f'data/10_cna/{celltype_major}.pkl', 'rb'))

    fig, axes = plt.subplots(figsize=(8, 1.1), ncols=6, nrows=1)
    axs = axes.flatten()
    plt.subplots_adjust(hspace=0, wspace=0.25)
    
    for idx, covar in enumerate(['race', 'group', 'age', 'diabetes', 'obesity', 'celltype_level3']):
        ax = axs[idx]
        covar_newname = covar if covar != 'group' else 'stage'
        covar_newname = covar if covar != 'group' else 'stage'
        coef_col = f"{covar_newname}_coef"
        fdr_col = f"{covar_newname}_coef_fdr"
        color_col = f"{covar_newname}_coef_plot"
    
        if covar == 'celltype_level3':
            cell_colors = {x: palette_all[x] for x in set(d.obs['celltype_level3'])}
            palette = {}
            for ct in d.obs['celltype_level3'].unique():
                if ct in cell_colors:
                    palette[ct] = cell_colors[ct]
                else:
                    palette[ct] = (0.8, 0.8, 0.8)
            sc.pl.umap(
                d,
                color='celltype_level3',
                palette=palette,
                size=.1,
                ax=ax,
                legend_fontsize='large',
                show=False,
                title=''
            )
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(
                handles,
                labels,
                loc='center left',
                bbox_to_anchor=(1.01, 0.5),  # right outside the last subplot
                fontsize='large',
                frameon=False
            )
        else:
            p = d.uns[f'cnaGlobalPval_{covar_newname}']
            d.obs[color_col] = np.where(d.obs[fdr_col] < 0.1, d.obs[coef_col], np.nan)   # nan will show as gray in sc.pl.umap
            
            # show_status = True if covar=='race' else False
            sc.pl.umap(
                d,
                color=color_col,
                s=.5,
                ax=ax,
                vmin=-1,
                vmax=1,
                cmap='seismic',
                na_color='lightgray',   # Gray for nan (non-significant)
                # colorbar_loc='',
                title='',
                show=False         # So you can further edit if needed
            )
            
            # Remove colorbar(s) from the figure after plotting
            # for extra_ax in ax.figure.axes:
            #     pos = extra_ax.get_position()
            #     # Colorbars are usually much narrower than main axes
            #     if (extra_ax != ax) and (pos.width < 0.1 or pos.height < 0.1):
            #         ax.figure.delaxes(extra_ax)
                    
        if celltype_major in ['B', 'NK', 'CD4 T', 'Mono']:
            x = 0.01
            ha_val = 'left'
        else:
            x = 0.99
            ha_val = 'right'
        
        ax.text(
            x, 0.01,
            f"p = {p:.3f}",
            ha=ha_val, va='bottom',
            fontsize=6,
            transform=ax.transAxes
        )
        ax.set_xlabel('')
        ax.set_ylabel('')
    
    plt.savefig(f'plots/Figure5/cna_{celltype_major}.svg')

# Replot, association table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# -----------------------------
# 1) Load and preprocess table
# -----------------------------
tbl = pd.read_csv('data/Figure5/CNA all cells table.csv', header=[0, 1], index_col=0)
tbl

In [ ]:
# Standardize column level-0 keys to lowercase to match palette keys
lvl0 = [str(c[0]).strip().lower() for c in tbl.columns]
lvl1 = [str(c[1]).strip().lower() for c in tbl.columns]
tbl.columns = pd.MultiIndex.from_arrays([lvl0, lvl1])

# Map 'Y'/'N' -> 1/0
tbl_bin = tbl.applymap(lambda x: 1 if str(x).strip().upper() == 'Y' else 0).astype(int)

# ---------------------------------
# 2) Palettes
# ---------------------------------
group_palette = {
    'race':     '#64a3e0',
    'stage':    '#f29d5c',
    'age':      '#66c2a5',
    'diabetes': '#F472B6',
    'obesity':  '#F59E42',
}
scope_palette = {'global': '#C9C9C9', 'local': '#9E9E9E'}
heatmap_cmap = ListedColormap(['#EEEEEE', '#F6B88A'])  # N, Y

# Column meta
col_vars  = tbl_bin.columns.get_level_values(0).tolist()
col_scope = tbl_bin.columns.get_level_values(1).tolist()
n_rows, n_cols = tbl_bin.shape

# Helper: consecutive runs (start/end) of identical variable names
def runs(seq):
    out = []
    start = 0
    for i in range(1, len(seq) + 1):
        if i == len(seq) or seq[i] != seq[start]:
            out.append((seq[start], start, i - 1))  # (value, start_idx, end_idx)
            start = i
    return out

# ---------------------------------
# 3) Figure layout (3 stacked axes)
# ---------------------------------
fig = plt.figure(figsize=(6, 3.3))
gs  = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[0.35, 0.35, 2.6], hspace=0.05)

ax_var   = fig.add_subplot(gs[0, 0])  # variable bar
ax_scope = fig.add_subplot(gs[1, 0])  # scope bar
ax_main  = fig.add_subplot(gs[2, 0])  # main heatmap

# ---------------------------------
# 4) Top annotation bars
# ---------------------------------
# Row 1: variable color blocks (per column), but label ONCE per run
ax_var.imshow(
    np.arange(n_cols)[None, :],
    aspect='auto',
    cmap=ListedColormap([group_palette[v] for v in col_vars]),
    vmin=-0.5, vmax=n_cols - 0.5
)

for val, s, e in runs(col_vars):
    x_center = (s + e) / 2.0
    ax_var.text(x_center, 0, val.capitalize(), ha='center', va='center',
                fontsize=8, color='black')

# Row 2: scope color blocks and per-column labels
scope_colors = [scope_palette[s] for s in col_scope]
ax_scope.imshow(np.arange(n_cols)[None, :], aspect='auto',
                cmap=ListedColormap(scope_colors),
                vmin=-0.5, vmax=n_cols - 0.5)
for j, s in enumerate(col_scope):
    ax_scope.text(j, 0, s.capitalize(), ha='center', va='center',
                  fontsize=8, color='black')

# Clean up annotation axes
for ax in (ax_var, ax_scope):
    ax.set_yticks([]); ax.set_xticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

# ---------------------------------
# 5) Main heatmap
# ---------------------------------
im = ax_main.imshow(tbl_bin.values, aspect='auto', cmap=heatmap_cmap, vmin=0, vmax=1)

ax_main.set_yticks(np.arange(n_rows))
ax_main.set_yticklabels(tbl_bin.index, fontsize=9)
ax_main.set_xticks(np.arange(n_cols))
ax_main.set_xticklabels([''] * n_cols)
ax_main.tick_params(top=False, bottom=False)

# Gridlines
ax_main.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
ax_main.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
ax_main.grid(which='minor', color='white', linewidth=0.6)
ax_main.tick_params(which='minor', bottom=False, left=False)

for sp in ax_main.spines.values():
    sp.set_visible(False)

# ---------------------------------
# 6) Thick separators every two columns (all three panels)
# ---------------------------------
def draw_separators(ax, n_cols, every=2, lw=1.8):
    for k in range(every, n_cols, every):
        ax.axvline(k - 0.5, color='black', linewidth=lw)

for ax in (ax_var, ax_scope, ax_main):
    draw_separators(ax, n_cols=n_cols, every=2, lw=1.8)

# ---------------------------------
# 7) Save
# ---------------------------------
plt.tight_layout()
plt.savefig('plots/Figure5/CNA_all_cells_binary_heatmap_annotated.svg', bbox_inches='tight')
plt.show()